# Session 2: How Models Learn Patterns

## The Story

You're the head of a motor insurance division. Your team has built a model to predict which customers will **lapse** (not renew). Let's see how a decision tree and a logistic regression actually learn — step by step.

This isn't a coding exercise. Follow the narrative, look at the output, and ask: **does this make sense?**

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Chapter 1: The Data

We have 5 years of policy renewals. Each row is a customer at renewal time.

In [ ]:
# Simulate a realistic lapse dataset
n = 5000

data = pd.DataFrame({
    'customer_age': np.random.normal(45, 13, n).clip(18, 80).astype(int),
    'tenure_years': np.random.exponential(4, n).clip(0, 20).astype(int),
    'annual_premium': np.random.lognormal(6.5, 0.4, n).clip(300, 3000).astype(int),
    'claims_last_3y': np.random.poisson(0.5, n),
    'premium_increase_pct': np.random.normal(5, 8, n).round(1),
    'calls_to_service': np.random.poisson(1.5, n),
    'has_multi_policy': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    'online_account': np.random.choice([0, 1], n, p=[0.45, 0.55]),
})

# Lapse depends on real factors
lapse_score = (
    0.3 * (data['premium_increase_pct'] > 10).astype(float) +
    0.2 * (data['tenure_years'] < 2).astype(float) +
    0.15 * (data['claims_last_3y'] == 0).astype(float) +  # No-claim customers shop around
    0.15 * (data['has_multi_policy'] == 0).astype(float) +
    0.1 * (data['calls_to_service'] > 3).astype(float) +
    0.1 * (data['customer_age'] < 30).astype(float) +
    np.random.normal(0, 0.25, n)
)
data['lapsed'] = (lapse_score > np.percentile(lapse_score, 85)).astype(int)

print("📋 CUSTOMER RENEWAL DATASET")
print("=" * 50)
print(f"  Customers: {len(data):,}")
print(f"  Lapse rate: {data['lapsed'].mean():.1%}")
print(f"  Features: {len(data.columns) - 1}")
print(f"\nFirst 5 rows:")
print(data.head().to_string(index=False))

## Chapter 2: The Decision Tree Learns

A decision tree asks yes/no questions to split the data. Let's watch it work:

In [ ]:
features = ['customer_age', 'tenure_years', 'annual_premium', 'claims_last_3y',
            'premium_increase_pct', 'calls_to_service', 'has_multi_policy', 'online_account']

X = data[features]
y = data['lapsed']

# Build a SHALLOW tree (max 3 levels) so we can read it
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

print("🌳 DECISION TREE (3 levels deep)")
print("=" * 60)
print(export_text(tree, feature_names=features, decimals=1))

In [ ]:
# Interpret the tree
importances = pd.Series(tree.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax, color='#00008f', alpha=0.8)
ax.set_title('Decision Tree: Which Features Matter Most?', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance (how much each feature helps separate lapsers from stayers)')
plt.tight_layout()
plt.show()

print("\n💡 The tree's first question (most important split) is about:")
print(f"   {importances.index[-1]} — this single feature separates lapsers best.")
print(f"\n   Read the tree above like a flowchart:")
print(f"   Start at the top, follow yes/no until you reach a leaf.")
print(f"   Each leaf gives a lapse prediction.")

### 🤔 Discussion

- Does the tree's logic match your business intuition?
- Would you trust this for customer retention decisions?
- What information is the tree MISSING that you know matters?

---

## Chapter 3: The Linear Model Learns

Instead of questions, a logistic regression assigns WEIGHTS to each feature:

In [ ]:
# Build logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X, y)

# Show the weights
weights = pd.Series(lr.coef_[0], index=features).sort_values(ascending=False)

print("📊 LOGISTIC REGRESSION: Feature Weights")
print("=" * 55)
print(f"\n  {'Feature':<25} {'Weight':>8}  {'Direction'}")
print("  " + "-" * 55)
for feat, w in weights.items():
    direction = "→ INCREASES lapse risk" if w > 0.05 else ("→ DECREASES lapse risk" if w < -0.05 else "→ minimal effect")
    print(f"  {feat:<25} {w:>+8.3f}  {direction}")

print(f"\n💡 Interpretation:")
print(f"   Positive weight → higher value = higher lapse risk")
print(f"   Negative weight → higher value = lower lapse risk")

In [ ]:
# Side-by-side: what both models say about specific customers
examples = pd.DataFrame({
    'customer_age': [25, 55, 35, 60],
    'tenure_years': [1, 12, 3, 8],
    'annual_premium': [950, 600, 800, 500],
    'claims_last_3y': [0, 0, 2, 1],
    'premium_increase_pct': [15.0, 3.0, 12.0, 2.0],
    'calls_to_service': [4, 0, 1, 1],
    'has_multi_policy': [0, 1, 0, 1],
    'online_account': [1, 0, 1, 1],
}, index=['Young, new, unhappy', 'Loyal veteran', 'Mid, recent increase', 'Stable multi-policy'])

tree_probs = tree.predict_proba(examples[features])[:, 1]
lr_probs = lr.predict_proba(examples[features])[:, 1]

print("📊 WHAT BOTH MODELS SAY ABOUT 4 CUSTOMERS")
print("=" * 65)
print(f"\n  {'Customer':<25} {'Tree P(lapse)':<16} {'Linear P(lapse)':<16} {'Agree?'}")
print("  " + "-" * 65)
for i, (name, tp, lp) in enumerate(zip(examples.index, tree_probs, lr_probs)):
    agree = "✅" if (tp > 0.5) == (lp > 0.5) else "⚠️ Disagree"
    print(f"  {name:<25} {tp:<16.1%} {lp:<16.1%} {agree}")

print(f"\n💡 Models often agree on clear cases but disagree on edge cases.")
print(f"   When they disagree, it's a signal to look more closely.")

## Chapter 4: Feature Engineering — The Real Leverage

Let's see what happens when we create BETTER features from the same raw data:

In [ ]:
# Create engineered features
data_enhanced = data.copy()
data_enhanced['premium_per_year_age'] = (data['annual_premium'] / data['customer_age']).round(1)
data_enhanced['claims_per_tenure'] = np.where(
    data['tenure_years'] > 0, 
    (data['claims_last_3y'] / data['tenure_years']).round(2), 
    data['claims_last_3y']
)
data_enhanced['high_increase_new'] = ((data['premium_increase_pct'] > 10) & (data['tenure_years'] < 3)).astype(int)
data_enhanced['engagement_score'] = data['online_account'] + data['has_multi_policy'] + (data['calls_to_service'] > 0).astype(int)

# Compare model with and without engineered features
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

features_enhanced = features + ['premium_per_year_age', 'claims_per_tenure', 'high_increase_new', 'engagement_score']
X_enh = data_enhanced[features_enhanced]
X_enh_train, X_enh_test, _, _ = train_test_split(X_enh, y, test_size=0.3, random_state=42)

# Basic model
lr_basic = LogisticRegression(max_iter=1000, random_state=42)
lr_basic.fit(X_train, y_train)
auc_basic = roc_auc_score(y_test, lr_basic.predict_proba(X_test)[:, 1])

# Enhanced model
lr_enhanced = LogisticRegression(max_iter=1000, random_state=42)
lr_enhanced.fit(X_enh_train, y_train)
auc_enhanced = roc_auc_score(y_test, lr_enhanced.predict_proba(X_enh_test)[:, 1])

print("📊 IMPACT OF FEATURE ENGINEERING")
print("=" * 50)
print(f"\n  Basic model (raw features only):     AUC = {auc_basic:.3f}")
print(f"  Enhanced model (+ engineered features): AUC = {auc_enhanced:.3f}")
print(f"  Improvement:                           +{(auc_enhanced - auc_basic):.3f}")
print(f"\n  Same algorithm. Same data. Better FEATURES → better results.")
print(f"\n  Engineered features added:")
print(f"    • premium_per_year_age  — normalizes premium by age")
print(f"    • claims_per_tenure     — claim intensity relative to time with us")
print(f"    • high_increase_new     — big price hike + short tenure (explosive combo)")
print(f"    • engagement_score      — multi-policy + online + has called us")
print(f"\n  💡 This is why 80% of DS time goes into data preparation.")
print(f"     Algorithms are commodity; features are competitive advantage.")

---

## Key Takeaways

1. **Decision trees** ask yes/no questions — readable but can be unstable
2. **Linear models** assign weights — stable but assume linear relationships
3. **Both learn from examples** — and are only as good as the data they see
4. **Feature engineering is the secret weapon** — same algorithm, better features = better model
5. **When models disagree, investigate** — edge cases deserve human judgment